In [1]:
import torch
import torch.nn as nn

import pandas as pd
import os
import json

import datascript
from datascript import load_data, split_data
import train as t

from model import LSTMModel
from tuning import tunemodel

import ray
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


Initialize Ray

In [2]:
ray.shutdown()  # Shutdown any existing Ray instances
ray.init(ignore_reinit_error=True)

Usage stats collection is enabled. To disable this, run the following command: `ray disable-usage-stats` before starting Ray. See https://docs.ray.io/en/master/cluster/usage-stats.html for more details.


2025-03-30 18:51:18,668	INFO worker.py:1807 -- Started a local Ray instance. View the dashboard at 127.0.0.1:8265 


Python version:,3.9.20
Ray version:,3.0.0.dev0
Dashboard:,http://127.0.0.1:8265


In [3]:
train_criterion = nn.MSELoss()  
test_criterion = t.RMSELoss()

In [4]:
specific= False
target_labels=["GWAP","LWAP"] 
regionnames=["LUZ"]
islands = ["Luzon","Visayas","Mindanao"]
model='LSTM'

In [5]:
epoch = 200
trials = 200
output_base_folder = os.path.dirname(os.path.abspath(datascript.__file__)) 
plotfolder = os.path.join(output_base_folder, "Graphs")
predsfolder= os.path.join(output_base_folder, "Preds",model)

Use Best Config as starting point

In [6]:
with open("finalconfig.json", "r") as file:
        startconfig = json.load(file)

In [7]:
startconfig["LUZGWAP"]["lag"]=1

In [9]:
transformed_features = load_data("LUZ", "GWAP", model, True, True)

Tune 

In [ ]:
config_dict = {}  # Dictionary to store best configurations

if not specific:
    for regionname in regionnames:
        island = "Luzon" if regionname == "LUZ" else "Visayas" if regionname == "VIS" else "Mindanao"
        
        for target_label in target_labels:
            key = f"{regionname}{target_label}"  # Example: "LUZLWAP", "VISGWAP", etc.
            config_filtered = {k: v for k, v in startconfig[key].items() if k != "name"}
            transformed_features = load_data(regionname, target_label, model, True, True)
            transformed_target = load_data(regionname, target_label, model, False, True)

            input_size = transformed_features.shape[1]*2 
            output_size = 1
            
            best_config = tunemodel(
                transformed_features, transformed_target,
                input_size, output_size, epoch, trials, 
                LSTMModel, train_criterion,initialconfig=[config_filtered]
            )

            
            for col in transformed_features.columns:
                transformed_features[col + "_LAGGED"] = transformed_features[col].shift(best_config["lag"]).fillna(0)
            
            X = transformed_features.values
            y = transformed_target.values

            actual_values = load_data(regionname, target_label, model, False, False)

            train_data, val_data, test_data = split_data(X)
            train_labels, val_labels, test_labels = split_data(y)

            all_metrics, predictions = t.run_hyperparams(best_config, train_data, val_data, test_data, 
            train_labels, val_labels,test_labels,input_size, output_size, epoch, island, LSTMModel, train_criterion, test_criterion, regionname, target_label,plotfolder)
            
            # Add the key inside best_config
            best_config["name"] = key  

            config_dict[key] = best_config
    

2025-03-30 18:51:21,423	INFO tune.py:616 -- [output] This uses the legacy output and progress reporter, as Jupyter notebooks are not supported by the new engine, yet. For more information, please see https://github.com/ray-project/ray/issues/36949
[I 2025-03-30 18:51:21,426] A new study created in memory with name: optuna


== Status ==
Current time: 2025-03-30 18:51:21 (running for 00:00:00.12)
Using AsyncHyperBand: num_stopped=0
Bracket: Iter 200.000: None | Iter 100.000: None | Iter 50.000: None
Logical resource usage: 5.0/20 CPUs, 0.25/1 GPUs (0.25/1.0 accelerator_type:G)
Result logdir: C:/Users/PAULOJ~1/AppData/Local/Temp/ray/session_2025-03-30_18-51-16_358902_16848/artifacts/2025-03-30_18-51-21/raytrain_2025-03-30_18-51-21/driver_artifacts
Number of trials: 1/200 (1 PENDING)
+-------------------+----------+-------+-----------------+------------------+--------------+-----------+----------+---------------+-------+--------------+------------+-------------------+--------------+------------+--------+-----------+----------------+
| Trial name        | status   | loc   | activation_fn   | activation_fn1   |   batch_size |   dropout |   factor |   hidden_size |   lag |   lambda_reg |         lr | norm_layer_type   |   num_layers |   patience |   seed |   seq_len |   weight_decay |
|-------------------+-----

2025-03-30 18:51:23,633	WARNING tune.py:219 -- Stop signal received (e.g. via SIGINT/Ctrl+C), ending Ray Tune run. This will try to checkpoint the experiment state one last time. Press CTRL+C (or send SIGINT/SIGKILL/SIGTERM) to skip. 
2025-03-30 18:51:23,637	INFO tune.py:1009 -- Wrote the latest version of all result files and experiment state to 'C:/Users/Paulo John Mercado/ray_results/raytrain_2025-03-30_18-51-21' in 0.0040s.


== Status ==
Current time: 2025-03-30 18:51:23 (running for 00:00:02.20)
Using AsyncHyperBand: num_stopped=0
Bracket: Iter 200.000: None | Iter 100.000: None | Iter 50.000: None
Logical resource usage: 5.0/20 CPUs, 0.25/1 GPUs (0.25/1.0 accelerator_type:G)
Result logdir: C:/Users/PAULOJ~1/AppData/Local/Temp/ray/session_2025-03-30_18-51-16_358902_16848/artifacts/2025-03-30_18-51-21/raytrain_2025-03-30_18-51-21/driver_artifacts
Number of trials: 1/200 (1 PENDING)
+-------------------+----------+-------+-----------------+------------------+--------------+-----------+----------+---------------+-------+--------------+------------+-------------------+--------------+------------+--------+-----------+----------------+
| Trial name        | status   | loc   | activation_fn   | activation_fn1   |   batch_size |   dropout |   factor |   hidden_size |   lag |   lambda_reg |         lr | norm_layer_type   |   num_layers |   patience |   seed |   seq_len |   weight_decay |
|-------------------+-----

2025-03-30 18:51:25,097	INFO tune.py:1041 -- Total run time: 3.67 seconds (2.19 seconds for the tuning loop).
2025-03-30 18:51:25,098	WARNING tune.py:1056 -- Experiment has been interrupted, but the most recent state was saved.
Resume experiment with: Tuner.restore(path="C:/Users/Paulo John Mercado/ray_results/raytrain_2025-03-30_18-51-21", trainable=...)
2025-03-30 18:51:25,100	WARNING experiment_analysis.py:180 -- Failed to fetch metrics for 1 trial(s):
- raytrain_f8a1dad4: FileNotFoundError('Could not fetch metrics for raytrain_f8a1dad4: both result.json and progress.csv were not found at C:/Users/Paulo John Mercado/ray_results/raytrain_2025-03-30_18-51-21/trial_f8a1dad4')


TypeError: 'NoneType' object is not subscriptable

Save a Config

In [ ]:
if os.path.exists(r'config.json'):
    with open("config.json", "r") as file:
        config = json.load(file)
else:
    with open("config.json", "w") as file:
        json.dump(config_dict, file)

Load Final Config

In [13]:
with open("finalconfig.json", "r") as file:
        finalconfig = json.load(file)

TEST


Run specific set of hyperparameter

In [10]:
#specific=True

In [11]:
if specific:
        for regionname in regionnames:
                island = "Luzon" if regionname == "LUZ" else "Visayas" if regionname == "VIS" else "Mindanao"
                for target_label in target_labels:
                        epoch=200
                        specific_config=startconfig[f"{regionname}{target_label}"]
                        transformed_features = load_data(regionname,target_label,model,True,True)
                        transformed_target = load_data(regionname,target_label,model,False,True)
                        transformed_features["LAGGED"]= transformed_features[target_label].shift(1).fillna(0)
                        X = transformed_features.values
                        y = transformed_target.values
                        train_data, val_data, test_data = split_data(X) 
                        train_labels, val_labels, test_labels = split_data(y)
                        actual_values = load_data(regionname,target_label,model,False,False)
                        input_size = train_data.shape[1]  
                        output_size = train_labels.shape[1]
                        _, predictions = t.run_hyperparams(specific_config, train_data, val_data, test_data, 
                        train_labels, val_labels,test_labels,input_size, output_size, epoch, island, LSTMModel, train_criterion, test_criterion, regionname, target_label,plotfolder)
                        predictions_df = pd.DataFrame(predictions)

                        csv_filename = f"{predsfolder}/{regionname}-{target_label}-predictions.csv"

                        # Save DataFrame to CSV
                        predictions_df.to_csv(csv_filename, index=False)